# 1.2 TileLang初识

## TileLang是什么

**TileLang（Tile Language）是一门用于简化高性能计算kernel开发的领域专用语言（DSL）。**

它采用Python语法，让开发者描述数据怎样搬运、执行什么计算，再由编译器生成设备上的程序。

本课程用TileLang在昇腾950上编写自定义算子。先从一个熟悉的场景开始：当PyTorch已经能完成计算时，我们为什么还要学习算子开发？


## 1. 从调用算子，到实现算子

在PyTorch中，让两个同形状的张量相加，写下`c = a + b`即可。框架会调用已有的加法算子，我们通常只需关心输入和结果。

现在设想，你想把几步计算组合起来，或者针对经常使用的数据形状重新安排计算过程。这时，关注的就不只是“算出什么”，还包括**数据怎样分配、计算怎样组织、结果怎样交给后续步骤**。这些原本藏在算子内部的工作，需要由开发者来安排。

TileLang提供了一种进入这个过程的方式：用Python语法写下设备侧的计算安排。你仍然要理解数据和硬件，但可以从熟悉的函数、循环与索引出发，逐步学会描述自己的算子。


## 2. 为什么选择TileLang

为什么选择TileLang来表达设备计算？下面这四个特点，可以帮助我们理解它的优势。

**采用Python语法，便于编写和修改。**

算子描述、输入准备和结果校验可以放在同一份Python程序中。熟悉Python的读者能够沿用已有的编程习惯，逐步认识用于设备计算的接口。

**以数据块为单位组织计算。**

开发者可以围绕“一次处理哪一部分输入”来安排搬运与计算：取出一部分数据，完成这部分工作，再写回结果。下一节结合硬件介绍这种数据块，也就是TileLang名称中的tile。

**让编译器承担部分底层工作。**

开发者描述数据搬运和计算操作，编译器将它们转换成所选后端的设备代码，并处理所支持程序所需的同步等细节。这减少了初次实现时需要直接编写的底层代码。

**保留面向硬件调整实现的空间。**

每次处理多少数据、数据放在哪里、采用哪些向量操作，仍然可以由开发者选择。随着学习深入，我们可以调整这些安排，检查结果并观察实际执行效果。

这四点结合起来，使我们能够用较熟悉的写法开始开发，同时逐步掌握影响设备执行的细节。

### TileLang 与 Ascend C：易用开发与极致控制

TileLang 和 Ascend C 都能面向 Ascend NPU 编写算子，但侧重点不同：TileLang 强调更高层的编程抽象、精简代码和快速开发调试；Ascend C 强调对硬件细节的全面控制和性能极致优化。

| 比较维度 | TileLang | Ascend C |
|---|---|---|
| **核心优势** | 使用 Python 内嵌 DSL，以 tile 为中心表达任务划分、数据搬运和计算，编程抽象更高，代码量更小，便于快速修改并与参考计算衔接。 | 直接使用设备编程接口，能够更灵活、细致地控制 GM/UB、队列、同步和计算过程，对性能进行极致优化。 |
| **开发与验证** | Host 代码、kernel 描述和结果校验可以留在 Python 工作流中，支持 JIT 编译，可以快速开发和调试。 | 设备实现与 Host 调用边界更明确，算子工程使开发交付件更标准化，质量控制更严格。 |
| **性能控制** | 可以调整 tile、存储位置、数据搬运和向量操作，可以通过底层暴露的硬件细节进行精细化的硬件控制和优化。 | 可以控制所有硬件细节，实现更灵活和极致的性能优化。 |
| **主要代价** | 受当前后端已支持的语法、数据类型和算子能力约束；使用特殊硬件能力时需要确认后端是否已暴露相应接口。 | 需要开发者处理更多设备细节，代码量、硬件知识要求、编译调试和维护成本通常更高。 |
| **更适合** | 新算子原型、融合思路验证、tile 策略探索，以及需要频繁迭代的实现。 | 需要极细控制硬件，实现性能极致优化，标准化算子交付。 |

Ascend C 也提供不同层级的 API；使用高层 API 时不一定要手动展开每一个底层步骤。这里的“控制所有硬件细节”强调的是：当优化需要继续下沉时，开发者能够直接使用设备接口控制关键硬件路径。

理解上述差异后，可以参考下图对两种开发方式做初步选择。

![TileLang 易用开发与 Ascend C 极致控制对比](images/01.02_paradigms_v3.png)

<p class="course-note"><small>TileLang 用更高层的抽象和更少代码提升开发效率；Ascend C 以更高的实现复杂度换取对硬件细节的全面控制、极致性能优化和标准化算子交付。</small></p>


## 3. TileLang的应用场景

了解了TileLang的特点，我们再来看它能用在哪些地方。什么时候值得自己写一个算子？下面三种需求，与我们后面的课程练习直接相关。

**想实现所需的计算逻辑。**

例如，编写Softmax、RMSNorm或GELU算子，或者尝试自定义的激活函数与归一化规则。

可以先用PyTorch写出用于检查结果的计算，再用TileLang实现设备侧程序。算法描述和结果校验都留在熟悉的Python环境中，便于反复修改和对照。

**想把连续几步计算放到一起。**

例如，在Attention（注意力计算）中，先做矩阵乘法，再经过缩放和Softmax，最后还要进行一次矩阵乘法。

开发者可以在一个算子里组织这些步骤，安排中间数据怎样传递和复用。TileLang对数据块、存储位置和计算操作的表达，有助于把这些安排写清楚。第5章将结合融合算子示例展开讲解。

**想围绕常用的数据形状调整实现。**

例如，针对固定行宽的Softmax、RMSNorm，或者常用矩阵尺寸的GEMM（矩阵乘法），可以尝试不同的数据划分和搬运安排。

TileLang保留了这些选择，便于在结果正确的基础上比较实现；实际效果仍需要通过运行与测量判断。

下面这张图把三种需求放在一起：左侧关注新的计算规则，中间以简化Attention展示多步计算的衔接，右侧关注常用输入形状下的实现选择。

![TileLang的三种应用场景：自定义计算逻辑、简化Attention融合计算、针对常用形状调整实现](images/01.02_tilelang_use_cases_v2.png)

<p class="course-note"><small>图中Q、K、V是Attention的输入，d是Q和K的特征维度；P表示Softmax得到的权重，O表示输出。这里先关注计算步骤的组合。</small></p>

这些场景并非TileLang独有。它的吸引力在于：**熟悉Python的开发者可以较快开始表达新的算子，同时保留按硬件组织实现的能力。** 对已有的AscendC代码或组件，也可以继续采用相应的开发方式。

在本课程中，我们会从加法起步，逐步学习向量算子、矩阵乘法和融合计算。先进入[1.3「TileLang与昇腾950」](01.03_ascend_runtime.ipynb)，认识这些计算将要使用的硬件，再理解“按数据块组织”究竟是什么意思。
